# RoboVacuumDDPG — Results Analysis

Bar-Ilan *Vibe Coding & RL* workshop, **Assignment 5** (DDPG).

This notebook is a **read-only analysis** of the sealed 5-seed run. It is an
**SDK consumer** (CLAUDE.md §3 / spec §9.2): it imports **only**
`RoboVacuumSDK` plus the standard library and plotting — never `src.env`,
`src.model`, `src.ddpg`, or `src.services` directly.

It loads `results/metrics_summary.json` (committed), the per-seed
`results/history/seed_*.json` if present (heavy, git-ignored), and
`results/holdout_eval.json`, recomputes the per-seed table + across-seed
mean ± std, and displays the three deliverable figures.

> **Honest result (spec §10):** the agent learns coverage control on the
> training map `room_single` (~39 % coverage, +691 ± 443 reward) and **partially**
> generalizes to the held-out maps (apt_large 10.4 % +/- 5.7 %, office 17.9 % +/- 7.9 %, at a real collision cost). Full discussion in
> [`../docs/ANALYSIS.md`](../docs/ANALYSIS.md).

In [ ]:
import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

# Resolve the repo root whether the notebook runs from notebooks/ or the root.
ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# SDK-consumer rule (spec §9.2): the ONLY src import allowed in this notebook.
from src.sdk.sdk import RoboVacuumSDK  # noqa: E402

sdk = RoboVacuumSDK(str(ROOT / "config" / "config.yaml"))
TRAIN_MAP = sdk.cfg["maps"]["train"][0]
SEEDS = sdk.cfg["training"]["seeds"]
EPISODES = sdk.cfg["training"]["episodes"]
TAU = sdk.cfg["ddpg"]["tau"]
GAMMA = sdk.cfg["ddpg"]["gamma"]
print(f"train map = {TRAIN_MAP} | seeds = {SEEDS} | episodes/seed = {EPISODES}")
print(f"gamma = {GAMMA} | tau (Polyak) = {TAU}")

RESULTS = ROOT / "results"
FIGS = RESULTS / "figures"

## 1. Per-seed results + across-seed mean ± std

We report the **mean over each seed's final 20 episodes** (the greedy-quality
tail). The committed `results/metrics_summary.json` is the source of truth; if
the heavy per-seed histories are present locally we **recompute** the same
numbers from raw to prove they match.

In [ ]:
FINAL_WINDOW = 20

summary = json.loads((RESULTS / "metrics_summary.json").read_text(encoding="utf-8"))


def recompute_from_history(seed):
    """Recompute the final-20 tail from raw history, or None if not present."""
    fp = RESULTS / "history" / f"seed_{seed}.json"
    if not fp.exists():
        return None
    recs = json.loads(fp.read_text(encoding="utf-8"))[-FINAL_WINDOW:]
    return (
        round(statistics.mean(r["reward"] for r in recs), 1),
        round(statistics.mean(r["coverage"] for r in recs), 4),
    )


print(f"{'seed':>5} | {'reward':>9} | {'coverage':>9} | {'max R':>7} | source")
print("-" * 52)
rewards, covs = [], []
for seed in SEEDS:
    ps = summary["per_seed"][str(seed)]
    live = recompute_from_history(seed)
    src_tag = "history (recomputed)" if live else "metrics_summary.json"
    reward = live[0] if live else ps["final20_reward"]
    cov = live[1] if live else ps["final20_coverage"]
    rewards.append(reward)
    covs.append(cov)
    print(f"{seed:>5} | {reward:>9.1f} | {cov:>9.4f} | {ps['max_episode_reward']:>7.0f} | {src_tag}")

mean_r = statistics.mean(rewards)
std_r = statistics.pstdev(rewards)
mean_c = statistics.mean(covs)
print("-" * 52)
print(f"across-seed mean reward = {mean_r:.1f} +/- {std_r:.1f} (pstdev)")
print(f"across-seed mean coverage = {mean_c:.4f} (~{mean_c * 100:.0f}% of free cells / episode)")

## 2. Held-out generalization (a real but partial transfer gap)

Trained on `room_single` only, then evaluated **greedy** (no further learning)
on the `config.maps.holdout` plans. The policy **partially** generalizes
(~10–18 % coverage vs ~39 % on the training map) but at a real collision cost,
so it has not learned transferable wall-avoidance — full discussion in
[`../docs/ANALYSIS.md`](../docs/ANALYSIS.md).

In [ ]:
holdout = json.loads((RESULTS / "holdout_eval.json").read_text(encoding="utf-8"))

print(f"train-map coverage (across-seed) = {mean_c:.4f}\n")
print(f"{'held-out map':>12} | {'coverage (mean +/- 95% CI)':>26} | {'collisions':>10} | {'steps':>6}")
print("-" * 64)
for name, ev in holdout.items():
    cov = f"{ev['coverage_mean']:.4f} +/- {ev['coverage_ci']:.4f}"
    print(f"{name:>12} | {cov:>26} | {ev['collisions_mean']:>10.1f} | {ev['steps']:>6}")

best = max(ev["coverage_mean"] for ev in holdout.values())
print(f"\ncoverage drops ~{mean_c / best:.1f}x from the train map to the best held-out map "
      "(a partial gap, not a collapse).")
print("=> partial generalization at a real collision cost; "
      "map-conditioned state needed for full transfer.")

## 3. The three deliverable figures

Rendered by `scripts/render_learning_curve.py`, `scripts/render_critic_loss.py`,
`scripts/render_trajectory.py`. We display the committed PNGs directly.

In [ ]:
for fname, caption in [
    ("learning_curve.png", "Learning curve — reward vs episode, mean +/- 95% CI (5 seeds)."),
    ("critic_loss.png", "Critic loss — per-episode-mean MSE vs episode; bounded, no divergence."),
    ("trajectory.png", "Trajectory — trained seed-42 greedy sweep on room_single."),
]:
    path = FIGS / fname
    print(caption)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"  (missing: {path} — run the matching render script)")
    print()

# Sanity: matplotlib is importable so the notebook can also render live figures.
plt.close("all")

## 4. The DDPG math (for reference)

**Deterministic policy gradient** — maximize the expected return of the
deterministic actor $\mu_\theta$ through the critic $Q_\phi$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{s\sim\mathcal{D}}\Big[\nabla_a Q_\phi(s,a)\big|_{a=\mu_\theta(s)}\;\nabla_\theta \mu_\theta(s)\Big].$$

**Critic TD target** (computed from the *target* networks $\mu_{\theta'},\,Q_{\phi'}$):

$$y = r + \gamma\,(1-d)\,Q_{\phi'}\big(s',\,\mu_{\theta'}(s')\big),\qquad
\mathcal{L}(\phi) = \mathbb{E}\big[(Q_\phi(s,a) - y)^2\big].$$

**Polyak (soft) target update** with $\tau = 0.005$ — the slow target that keeps
the critic regression stationary (Q3):

$$\theta' \leftarrow \tau\,\theta + (1-\tau)\,\theta',\qquad
\phi' \leftarrow \tau\,\phi + (1-\tau)\,\phi'.$$

**Exploration** — additive Gaussian action noise, $a = \mu_\theta(s) + \mathcal{N}(0,\sigma^2)$,
with $\sigma$ decayed linearly $0.2 \to 0.05$.

### Citations
- Lillicrap, T. P. et al. (2016). *Continuous Control with Deep Reinforcement Learning* (DDPG). arXiv:1509.02971.
- Li, T. et al. (2019). *HouseExpo: A Large-scale 2D Indoor Layout Dataset.* arXiv:1903.09845.